# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Numerical Integration Schemes: Predictor-Corrector (PCEC), Adaptive RK45, Neural ODEs, Liquid Time-Constant (LTC), and Neural SDEs
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Lambert, J. D. (1991). *Numerical Methods for Ordinary Differential Systems*. John Wiley & Sons.
- Chen, R. T. Q., Rubanova, Y., Bettencourt, J., & Duvenaud, D. (2018). *Neural Ordinary Differential Equations*. NeurIPS.
- Hasani, R., Lechner, M., Amini, A., Rus, D., & Grosu, R. (2021). *Liquid Time-Constant Networks*. AAAI.
- Hasani, R., et al. (2022). *Closed-form Continuous-Time Neural Networks*. Nature Machine Intelligence.
- Li, X., Wong, T. K. L., Chen, R. T. Q., & Duvenaud, D. (2020). *Scalable Gradients for Stochastic Differential Equations*. AISTATS.

## 2. Mathematical Foundations & Governing Equations

### Numerical Integrator Zoo
- **Explicit Euler**: $x_{n+1} = x_n + h f(x_n)$
- **Heun's 2nd-Order**: $k_1 = f(x_n), \quad k_2 = f(x_n + h k_1), \quad x_{n+1} = x_n + \frac{h}{2}(k_1 + k_2)$
- **PCEC Predictor-Corrector**: Predicts with Adams-Bashforth and corrects with Adams-Moulton: $x_{n+1} = x_n + \frac{h}{2}(f_n + f_{n+1}^{(C)})$
- **Classical RK4**: $x_{n+1} = x_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$

### Continuous Adjoint Sensitivity Method
$$\frac{da(t)}{dt} = -a(t)^T \frac{\partial f(t, x(t))}{\partial x} \implies a(t - h) = a(t) + h \cdot a(t)^T J_f(x(t))$$

### Closed-Form Continuous-Time (CfC) & Neural SDE (Milstein)
$$x_{\text{CfC}}(t) \approx \sigma(-|f| t) \odot (x_0 W_1) + (1 - \sigma(-|f| t)) \odot (x_0 W_2)$$
$$x_{\text{Milstein}} = x_{\text{EM}} + \frac{1}{2} g(x_n) g'(x_n) \left( \Delta W^2 - \Delta t \right)$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
from safetensors.torch import load_file

torch.manual_seed(42)

# Initial state: B=2, D=2
x0 = torch.randn(2, 2)

# Neural ODE vector field parameters
w1 = torch.randn(2, 4) * 0.5
b1 = torch.randn(4) * 0.1
w2 = torch.randn(4, 2) * 0.5
b2 = torch.randn(2) * 0.1

def f_vec(x):
    z = x @ w1 + b1
    a = torch.tanh(z)
    return a @ w2 + b2

h = 0.1
f0 = f_vec(x0)
print(f"State x0: {x0.shape}, Vector field f0: {f0.shape}, Step size h: {h}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
# 1. Explicit Euler
x_euler = x0 + h * f0

# 2. Heun's 2nd-order (RK2)
k1 = f0
k2 = f_vec(x0 + h * k1)
x_heun = x0 + 0.5 * h * (k1 + k2)

# 3. PCEC Predictor-Corrector
x_p = x0 + h * f0
f_p = f_vec(x_p)
x_c1 = x0 + 0.5 * h * (f0 + f_p)
f_c = f_vec(x_c1)
x_pcec = x0 + 0.5 * h * (f0 + f_c)

# 4. Classical RK4
rk_k1 = f0
rk_k2 = f_vec(x0 + 0.5 * h * rk_k1)
rk_k3 = f_vec(x0 + 0.5 * h * rk_k2)
rk_k4 = f_vec(x0 + h * rk_k3)
x_rk4 = x0 + (h / 6.0) * (rk_k1 + 2.0 * rk_k2 + 2.0 * rk_k3 + rk_k4)

# 5. Continuous Adjoint Sensitivity
adj_T = torch.tensor([[1.0, -1.0], [0.5, 0.5]])
# Analytical Jacobian
z0 = x0 @ w1 + b1
a0 = torch.tanh(z0)
d_a0 = 1.0 - a0**2
j0 = torch.einsum("ik, bk, kj -> bij", w1, d_a0, w2)
adj_0 = adj_T + h * torch.einsum("bj, bij -> bi", adj_T, j0)

# 6. CfC Closed-Form Evaluation
w_cfc1 = torch.randn(2, 2) * 0.5
w_cfc2 = torch.randn(2, 2) * 0.5
gate = torch.sigmoid(-torch.abs(f0) * h)
x_cfc = gate * (x0 @ w_cfc1) + (1.0 - gate) * (x0 @ w_cfc2)

# 7. SDE Euler-Maruyama & Milstein
sigma = 0.2
dW = torch.randn(2, 2) * (h**0.5)
x_em = x0 + h * f0 + sigma * x0 * dW
x_milstein = x_em + 0.5 * (sigma**2) * x0 * (dW**2 - h)

print(f"Integrator outputs: Euler {x_euler.shape}, Heun {x_heun.shape}, PCEC {x_pcec.shape}, RK4 {x_rk4.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Local truncation error scaling: ||x_RK4 - x_Heun|| < ||x_Heun - x_Euler||
err_euler_heun = torch.norm(x_heun - x_euler).item()
err_heun_rk4 = torch.norm(x_rk4 - x_heun).item()
assert err_heun_rk4 < err_euler_heun, "Higher-order integrator did not show smaller relative correction"

# Invariant 2: Analytical vs Autograd Jacobian parity
x0_grad = x0.clone().detach().requires_grad_(True)
f0_auto = f_vec(x0_grad)
j_auto = torch.zeros(2, 2, 2)
for b in range(2):
    for j in range(2):
        grad_out = torch.zeros(2, 2)
        grad_out[b, j] = 1.0
        j_auto[b, :, j] = torch.autograd.grad(f0_auto, x0_grad, grad_outputs=grad_out, retain_graph=True)[0][b]
diff_j = torch.max(torch.abs(j0 - j_auto)).item()
assert diff_j < 1e-6, f"Jacobian autograd discrepancy: {diff_j:.2e}"
print(f"All mathematical invariants (truncation error hierarchy, analytical Jacobian diff {diff_j:.2e}) verified.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = f_vec(x0)
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"Vector field evaluation latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Explicit Euler on stiff linear oscillator diverges when h > 2 / lambda_max
k_stiff = 50.0 # lambda_max = 50
x_stiff = torch.tensor([[1.0, 0.0]])
h_unstable = 0.06 # h * lambda = 3.0 > 2.0 (outside stability domain)
x_div = x_stiff.clone()
for _ in range(20):
    # dx/dt = [x_2, -k*x_1]
    f_stiff = torch.cat([x_div[:, 1:2], -k_stiff * x_div[:, 0:1]], dim=-1)
    x_div = x_div + h_unstable * f_stiff
div_norm = torch.norm(x_div).item()
print(f"Pathological Euler 20-step explosion norm on stiff system: {div_norm:.2e}")

# Corrected: RK4 with smaller adaptive step or Predictor-Corrector maintains stable bounded trajectory
x_stable = x_stiff.clone()
h_stable = 0.02 # h * lambda = 1.0 < 2.0
for _ in range(20):
    f_s = torch.cat([x_stable[:, 1:2], -k_stiff * x_stable[:, 0:1]], dim=-1)
    x_stable = x_stable + h_stable * f_s
print(f"Corrected stable trajectory norm: {torch.norm(x_stable).item():.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Stability Over Accuracy**: Multi-step predictor-corrector (PCEC) schemes expand the linear stability domain on stiff neural ODE dynamics without expensive matrix inversions.
2. **Memory Efficiency via Continuous Adjoints**: Adjoint sensitivity solves backward gradients in $\mathcal{O}(1)$ memory by integrating backward in time.
3. **O(1) Evaluation via CfC**: Closed-form continuous-time networks bypass iterative numerical solvers entirely, evaluating in constant time.